# Load epub book

In [2]:
!uv pip install chromadb

Using Python 3.12.13 environment at: /usr
Resolved 81 packages in 574ms
Prepared 10 packages in 966ms
Installed 10 packages in 46ms
 + bcrypt==5.0.0
 + build==1.5.0
 + chromadb==1.5.9
 + durationpy==0.10
 + kubernetes==36.0.2
 + onnxruntime==1.26.0
 + opentelemetry-exporter-otlp-proto-grpc==1.38.0
 + pybase64==1.4.3
 + pypika==0.51.1
 + pyproject-hooks==1.2.0


In [1]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import CrossEncoder

/tmp/ipykernel_8565/121917837.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredEPubLoader


In [3]:
# TODO: Load document
book = "/content/docs/charles-dickens_a-christmas-carol.epub"
chunk_size = 1024
chunk_overlap = 100

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


In [4]:
# TODO Split document
epub_loader = UnstructuredEPubLoader(file_path=book)
chunks = epub_loader.load_and_split(text_splitter)

[WARNING] Sandbox argument was used, but pandoc version is too low. Ignoring argument.


In [14]:
# TODO Examine chunk
print(len(chunks))
idx = 100
print(chunks[idx].page_content)
print(chunks[idx].metadata)


204
The house-fronts looked black enough, and the windows blacker, contrasting with the smooth white sheet of snow upon the roofs, and with the dirtier snow upon the ground; which last deposit had been ploughed up in deep furrows by the heavy wheels of carts and wagons: furrows that crossed and recrossed each other hundreds of times where the great streets branched off; and made intricate channels, hard to trace in the thick yellow mud and icy water. The sky was gloomy, and the shortest streets were choked up with a dingy mist, half thawed, half frozen, whose heavier particles descended in a shower of sooty atoms, as if all the chimneys in Great Britain had, by one consent, caught fire, and were blazing away to their dear heart’s content. There was nothing very cheerful in the climate or the town, and yet was there an air of cheerfulness abroad that the clearest summer air and brightest summer sun might have endeavoured to diffuse in vain.
{'source': '/content/docs/charles-dickens_a-ch

# Create embeddings

In [15]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"

embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
# TODO: Explore embedding model
text = 'hello, world'
text = 'Siri AI, which finally arrives two years after Apple first announced a Siri revamp, can now handle more complex and multi-step tasks. The company says its improved assistant can understand the context of whats on your screen, pull up relevant information across various apps and carry out a more natural back-and-forth conversation. Its designed to feel seamless, practical and actually helpful. No more vague "I found this on the web" replies (hopefully). '

embedding = embed_func.embed_query([ text ])
# print(embedding)
print(len(embedding[0]))
print(embedding)


384
[array([-6.92600235e-02, -3.75798903e-02,  3.11365202e-02, -3.89862582e-02,
       -3.91205624e-02,  6.89372187e-03,  5.94172738e-02,  1.74034573e-02,
        6.78631430e-03,  2.93863546e-02, -2.53764465e-02,  7.20915617e-03,
        5.46315238e-02,  2.20539588e-02,  6.18768185e-02,  1.11987218e-02,
        5.52085675e-02, -1.11754812e-01, -6.90681711e-02, -5.11192046e-02,
        2.03026515e-02,  3.28340866e-02, -1.99800190e-02,  1.63821932e-02,
       -4.56386022e-02,  3.54417562e-02, -3.98020186e-02, -3.98270711e-02,
       -2.81415936e-02, -1.38945445e-01,  3.60267982e-02,  2.21964438e-02,
        7.88681358e-02,  4.11077142e-02, -6.33195490e-02,  5.47153912e-02,
       -2.54175588e-02,  1.16485832e-02,  1.13760903e-02,  3.39299478e-02,
        1.09173972e-02, -1.27101634e-02, -4.35221009e-03, -1.21611571e-02,
        5.28076552e-02, -6.14851387e-03,  1.36471903e-02,  5.90940053e-03,
       -2.25253049e-02, -1.48758963e-02,  2.23553106e-02, -3.21824737e-02,
        2.17882395e-

In [23]:
# TODO: Prepare the chunks for inserting into Chroma
texts = [ c.page_content for c in chunks ]
print(len(texts))
print(texts[3])


204
The mention of Marley’s funeral brings me back to the point I started from. There is no doubt that Marley was dead. This must be distinctly understood, or nothing wonderful can come of the story I am going to relate. If we were not perfectly convinced that Hamlet’s father died before the play began, there would be nothing more remarkable in his taking a stroll at night, in an easterly wind, upon his own ramparts, than there would be in any other middle-aged gentleman rashly turning out after dark in a breezy spot﻿—say St. Paul’s Churchyard, for instance﻿—literally to astonish his son’s weak mind.

Scrooge never painted out Old Marley’s name. There it stood, years afterwards, above the warehouse door: Scrooge and Marley. The firm was known as Scrooge and Marley. Sometimes people new to the business called Scrooge Scrooge, and sometimes Marley, but he answered to both names. It was all the same to him.


In [26]:
text_ids = [ str(uuid4())[:8] for _ in range(len(texts)) ]
print(len(text_ids))
print(text_ids)

204
['2513ae47', 'e069aea7', 'b5b40ca8', 'e13559d6', '43b1602b', '49e80710', '9fdf1f98', '641ea351', '08328f14', 'a7636511', '3a4a4d8f', '9820d65a', 'da056b71', 'f5e25586', '33084db1', 'f5e2a837', '15d3e0aa', 'a0faddfd', 'efa95bb8', 'cde61053', '5d5d5789', 'a394fb16', '6a984b87', 'ffeb66e7', '881b91ad', 'd1acf759', '9121a2c6', 'b87e8cbd', '6724c880', 'e627c6eb', 'a8237bc2', 'ea4a74bd', 'bf80dbb2', '918c27db', '58e33711', 'bd67808e', '5a19b7e3', '86e700eb', '3b5dddb5', 'fd96a0ff', '1626c5a9', 'ec246465', '56a74af8', 'cde0d78d', '32138f14', 'f2bf1c1c', '7b0e2103', 'b97dc9e9', 'a6ae7ac5', '471f4d75', '76beb584', 'e40ad87d', '58da0614', '1be1a84c', 'e2fc9706', '1c1daa7e', '54cfa910', '63b115bd', 'd88115c7', 'c72f7370', 'b007a64b', '9eaa481f', '39516a76', '04ba338d', '0416b2a8', '327c16c9', '9e413d16', '85f993dc', 'b35548cc', '59e6ae93', '3cbfaf07', 'd20fb0f3', 'b27a3417', 'aa569cbe', 'aead9608', '99b8170b', '59dc6d6e', '90f419d5', '3a2f4702', '09a938fe', 'd4b42772', '59749fd5', 'c87182c4',

In [27]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = "christmas_carol"

# Create a ChromaDB client
chroma_client = chromadb.Client()

# Delete the collection if it already exists
try:
    chroma_client.delete_collection(name=col_name)
except:
    pass

# Create the collection
collection = chroma_client.create_collection(
    name=col_name,
    embedding_function=embed_func
)


In [28]:
# TODO: Print number of documents in collection
# Insert the chunks and the ids into the collection
collection.add(
    documents=texts,
    ids=text_ids
)


In [29]:
print(collection.count())

204


In [76]:
# TODO: Query collection
query = "Who was Scrooge's deceased business partner?" # correct
query = "How many ghosts visit Scrooge in total?" # wrong
query = "What is the name of Bob Cratchit's youngest son who is ill?" # correct

results = collection.query(
    query_texts=query,
    n_results=10
)

for k, v in results.items():
  print(f'{k}: {v}')

ids: [['71e3ae8f', 'a1a067bd', 'c1ac6110', '4e014656', '32db3322', '11de87b9', '6f005958', '669b8847', '350ce735', '52c2c73c']]
embeddings: None
documents: [['She hurried out to meet him; and little Bob in his comforter\ufeff—he had need of it, poor fellow\ufeff—came in. His tea was ready for him on the hob, and they all tried who should help him to it most. Then the two young Cratchits got upon his knees, and laid, each child, a little cheek against his face, as if they said, “Don’t mind it, father. Don’t be grieved!”\n\nBob was very cheerful with them, and spoke pleasantly to all the family. He looked at the work upon the table, and praised the industry and speed of Mrs. Cratchit and the girls. They would be done long before Sunday, he said.\n\n“Sunday! You went today, then, Robert?” said his wife.\n\n“Yes, my dear,” returned Bob. “I wish you could have gone. It would have done you good to see how green a place it is. But you’ll see it often. I promised him that I would walk there on

In [77]:
orig_order = []
for id in results['ids'][0]:
  doc = collection.get(ids=[id])
  orig_order.append(doc['documents'][0])
print(orig_order)
print(len(orig_order))

['She hurried out to meet him; and little Bob in his comforter\ufeff—he had need of it, poor fellow\ufeff—came in. His tea was ready for him on the hob, and they all tried who should help him to it most. Then the two young Cratchits got upon his knees, and laid, each child, a little cheek against his face, as if they said, “Don’t mind it, father. Don’t be grieved!”\n\nBob was very cheerful with them, and spoke pleasantly to all the family. He looked at the work upon the table, and praised the industry and speed of Mrs. Cratchit and the girls. They would be done long before Sunday, he said.\n\n“Sunday! You went today, then, Robert?” said his wife.\n\n“Yes, my dear,” returned Bob. “I wish you could have gone. It would have done you good to see how green a place it is. But you’ll see it often. I promised him that I would walk there on a Sunday. My little, little child!” cried Bob. “My little child!”', 'So Martha hid herself, and in came little Bob, the father, with at least three feet of 

# Cross Encoder
A Cross-Encoder is a model used to determine the semantic similarity, relevance, or relationship between two pieces of text (such as a search query and a document, or a premise and a hypothesis). It embeds 2 pieces of text **together**, allowing full word-by-word cross-attention.

A Bi-Encoder on the other hand embeds 2 pieces of text **independently** into vectors, then calculates similarity (e.g., Cosine Similarity).

In modern RAG application, we combine both bi-encoder and cross-encoder into a two-stage pipeline:

Stage 1: Retrieval (Bi-Encoder): A fast Bi-Encoder searches through millions of documents in a vector database and instantly narrows them down to the top 50 or 100 most likely candidates.

Stage 2: Re-ranking (Cross-Encoder): The slower, highly accurate Cross-Encoder evaluates only those top 50 or 100 candidates side-by-side with the user's query, re-sorting them to ensure the absolute most contextually relevant documents are placed at the very top.

In [43]:
# Use a model with only a single label for reranking
# Takes about approx 6 minutes to load
cross_encode_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"

cross_encoder = CrossEncoder(cross_encode_model)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [78]:
# Extract all text from the result
reranked_order = cross_encoder.rank(query, orig_order, top_k=3)
#print(reranked_order)
for i in reranked_order:
  print(i)

{'corpus_id': 0, 'score': np.float32(-2.0634727)}
{'corpus_id': 1, 'score': np.float32(-2.9864595)}
{'corpus_id': 3, 'score': np.float32(-3.3791516)}


In [79]:
# Rerank the documents
print('Query')
print(query)

context = ""

for i in reranked_order:
  idx = i['corpus_id']
  print(orig_order[idx])
  context = context + orig_order[idx]
  print('----------------------')

Query
What is the name of Bob Cratchit's youngest son who is ill?
She hurried out to meet him; and little Bob in his comforter﻿—he had need of it, poor fellow﻿—came in. His tea was ready for him on the hob, and they all tried who should help him to it most. Then the two young Cratchits got upon his knees, and laid, each child, a little cheek against his face, as if they said, “Don’t mind it, father. Don’t be grieved!”

Bob was very cheerful with them, and spoke pleasantly to all the family. He looked at the work upon the table, and praised the industry and speed of Mrs. Cratchit and the girls. They would be done long before Sunday, he said.

“Sunday! You went today, then, Robert?” said his wife.

“Yes, my dear,” returned Bob. “I wish you could have gone. It would have done you good to see how green a place it is. But you’ll see it often. I promised him that I would walk there on a Sunday. My little, little child!” cried Bob. “My little child!”
----------------------
So Martha hid herse

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved.

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file.

In [61]:
# TODO Your code
model_name = "google/flan-t5-base"

# Load the model and the tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [80]:
# TODO Your code
prompt = f"""Answer based on context:\n\n{context}\n\n{query}"""
print(prompt)

Answer based on context:

She hurried out to meet him; and little Bob in his comforter﻿—he had need of it, poor fellow﻿—came in. His tea was ready for him on the hob, and they all tried who should help him to it most. Then the two young Cratchits got upon his knees, and laid, each child, a little cheek against his face, as if they said, “Don’t mind it, father. Don’t be grieved!”

Bob was very cheerful with them, and spoke pleasantly to all the family. He looked at the work upon the table, and praised the industry and speed of Mrs. Cratchit and the girls. They would be done long before Sunday, he said.

“Sunday! You went today, then, Robert?” said his wife.

“Yes, my dear,” returned Bob. “I wish you could have gone. It would have done you good to see how green a place it is. But you’ll see it often. I promised him that I would walk there on a Sunday. My little, little child!” cried Bob. “My little child!”So Martha hid herself, and in came little Bob, the father, with at least three feet

In [81]:
# TODO Your code
enc_prompt = tokenizer(prompt, return_tensors='pt')

# Generate the output
enc_answer = model.generate(**enc_prompt)

# Decode the output
answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)

# Print the output
print(f"Question: {query}")
print(answer)

Question: What is the name of Bob Cratchit's youngest son who is ill?
Tiny Tim


# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?